# Notebook de estudio: Proyecto LACDA

**Para que sirve este notebook.** Es una guia de aprendizaje del proyecto completo, pensada para preparar la defensa oral del EP2. Aqui vas a encontrar, en orden:

1. Que es un *default* y que es un *pagado* (la pregunta que define todo el proyecto).
2. El diccionario de las 14 variables del dataset, ordenadas por tipo.
3. Como se modelan esas variables en 2 entidades (solicitante y prestamo) y por que.
4. El pipeline completo de 5 etapas (ingesta -> qualitycheck -> limpieza -> transformacion -> validacion), explicado paso a paso con ejemplos chicos.
5. De donde salieron las 3 features derivadas y por que no hay mas.
6. Un cierre con las preguntas frecuentes que pueden caer en la defensa.

**Como leerlo.** Las celdas markdown explican el *que* y el *por que*; las celdas de codigo muestran el *como* sobre una muestra chica del CSV. Todo corre sin Docker ni Postgres — solo lee `data/loan_data.csv`.

In [ ]:
# Setup minimo: cargamos el CSV crudo desde data/ y una muestra chica
from pathlib import Path
import numpy as np
import pandas as pd

CSV = Path().resolve().parent / "data" / "loan_data.csv"
df = pd.read_csv(CSV)

print(f"Filas totales: {len(df):,}")
print(f"Columnas: {df.shape[1]}")
df.head(3)

---
## 1. Lo primero: que es un *default* y que es un *pagado*

Toda la prediccion del proyecto gira en torno a una sola columna: **`loan_status`**, que vale 0 o 1.

| Valor | Significado en este dataset | Como se interpreta |
|---|---|---|
| **`loan_status = 1`** | El prestamo cayo en **default** | El cliente **no pago** el credito. Es la mala noticia. |
| **`loan_status = 0`** | El prestamo fue **pagado** | El cliente cumplio sus cuotas. Es la buena noticia. |

**Que es "default" en lenguaje financiero.** Un prestamo entra en default cuando el deudor deja de pagar segun lo pactado y el banco lo da por incobrable (en la practica suele ser ~90 dias sin pagar, pero en este dataset el etiquetado ya viene resuelto). Para el banco es perdida directa de dinero, por eso es lo que se quiere predecir antes de aprobar el credito.

**Por que importa.** Si predecimos bien que solicitantes caeran en default, el banco puede negar credito a los riesgosos y aprobar a los buenos pagadores. El modelo final (fuera del alcance de esta entrega) se entrena para responder esta pregunta: *dado lo que se del solicitante y del prestamo solicitado, ¿caera en default?*

**Como se reparte en nuestro dataset.** Lo vemos abajo: ~22% son defaults, ~78% pagados. Es un dataset **desbalanceado** — algo a tener en cuenta cuando se entrene el modelo (precision/recall importan mas que accuracy).

In [ ]:
dist = df["loan_status"].value_counts().sort_index()
pct = (dist / len(df) * 100).round(2)
pd.DataFrame({"casos": dist, "%": pct}).rename(
    index={0: "Pagado (loan_status=0)", 1: "Default (loan_status=1)"}
)

---
## 2. Diccionario de variables (por tipo)

El CSV trae **14 columnas**. Las dividimos por tipo de dato y, en cada tipo, las explicamos en castellano simple.

### 2.1. Numericas continuas (toman cualquier valor real dentro de un rango)

| Variable | Que mide | Unidad | Rango valido (cap. 9) |
|---|---|---|---|
| `person_age` | Edad del solicitante | años | 18 a 100 |
| `person_income` | Ingreso anual del solicitante | USD/año | >= 0 (sin tope duro) |
| `loan_amnt` | Monto del prestamo solicitado | USD | > 0 (sin tope duro) |
| `loan_int_rate` | Tasa de interes del prestamo | % anual | 5 a 30 |
| `loan_percent_income` | Cuanto del ingreso anual representa el prestamo | proporcion 0-1 | 0 a 1 |
| `cb_person_cred_hist_length` | Antiguedad del historial crediticio | años | 0 a `person_age` |

### 2.2. Numericas discretas (entero, conteo o escala)

| Variable | Que mide | Rango valido |
|---|---|---|
| `person_emp_exp` | Años de experiencia laboral | 0 a `person_age - 18` |
| `credit_score` | Puntaje crediticio tipo FICO | 300 a 850 |

### 2.3. Categoricas nominales (texto, conjunto cerrado de valores)

| Variable | Valores permitidos |
|---|---|
| `person_gender` | `male`, `female` |
| `person_education` | `High School`, `Associate`, `Bachelor`, `Master`, `Doctorate` |
| `person_home_ownership` | `RENT`, `OWN`, `MORTGAGE`, `OTHER` |
| `loan_intent` | `PERSONAL`, `EDUCATION`, `MEDICAL`, `VENTURE`, `DEBTCONSOLIDATION`, `HOMEIMPROVEMENT` |

### 2.4. Binarias

| Variable | Valores | Significado |
|---|---|---|
| `previous_loan_defaults_on_file` | `Yes` / `No` | ¿Tiene defaults previos en su historial? |
| `loan_status` | `1` / `0` | **El target.** 1 = default, 0 = pagado. |

### 2.5. Reglas cruzadas (consistencia entre columnas)

Estas no son columnas, pero son invariantes que tienen que cumplirse entre columnas:

- `person_emp_exp <= person_age - 18` (no se puede tener mas experiencia laboral que años "trabajables").
- `cb_person_cred_hist_length <= person_age` (el historial crediticio no puede ser mayor que la edad de la persona).

**Recordar:** estas dos reglas se aplican *despues* de fijar `person_age`, porque dependen de el. Si no se respeta el orden, la limpieza se rompe (bug ya detectado y corregido).

In [ ]:
# Resumen rapido de tipos de datos y nulos por columna
resumen = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),
    "nulos": df.isnull().sum(),
    "unicos": df.nunique(),
    "ejemplo": df.iloc[0].astype(str).values,
})
resumen

---
## 3. Dos entidades, no una: solicitante y prestamo

El CSV crudo tiene las 14 columnas mezcladas en una sola fila por prestamo, pero conceptualmente hay **dos cosas distintas** descritas ahi:

- **Solicitante (persona):** quien pide el credito. Sus atributos no dependen del prestamo (edad, genero, educacion, ingreso, experiencia, vivienda, historial, score).
- **Prestamo (operacion de credito):** los detalles del credito puntual (monto, motivo, tasa, % del ingreso, defaults previos, status).

Por eso el modelo fisico los **separa en dos tablas** unidas por una clave foranea (FK):

```
solicitantes_*  (id)  <---FK---  prestamos_*  (solicitante_id)
```

**Por que separar.** En el mundo real, una persona puede pedir varios prestamos. El dataset academico tiene 1-a-1 (cada fila es un solicitante con un prestamo), pero diseñar 1-a-N desde el principio es mas profesional y escalable. La rubrica EP2 lo pide explicitamente (cap. 8 del diseño tecnico).

### 3.1. Las 6 tablas de Postgres

El pipeline tiene 3 etapas que persisten datos, y cada etapa duplica el par de tablas:

| Etapa | Tabla solicitante | Tabla prestamo |
|---|---|---|
| Ingesta | `solicitantes_raw` | `prestamos_raw` |
| Limpieza | `solicitantes_clean` | `prestamos_clean` |
| Transformacion | `solicitantes_transformed` | `prestamos_transformed` |

Son 6 tablas en total. Cada una con la misma estructura base de su entidad, agregando lo que la etapa aporta. Mantener las 3 versiones permite **trazabilidad** (puedo reconstruir el dato original aunque limpieza lo haya tocado) y **debuggeo** (si algo falla en transformacion, comparo contra clean).

### 3.2. Como se mantiene la integridad referencial

Cuando se carga una etapa, primero se inserta la tabla padre (`solicitantes_*`), se recuperan los `id` autogenerados *en orden*, y se asignan como `solicitante_id` a la tabla hija (`prestamos_*`) por posicion. Esto solo funciona porque las filas se mantienen ordenadas — cualquier `sort` o `sample` intermedio rompe la FK silenciosamente.

---
## 4. Pipeline completo: las 5 etapas

```
  data/loan_data.csv
         |
         v
  (1) ingesta.py        -> solicitantes_raw, prestamos_raw
         |
         v
  (2) qualitycheck.py   -> KPIs sobre raw (informativo, no rompe)
         |
         v
  (3) limpieza.py       -> solicitantes_clean, prestamos_clean
         |
         v
  (4) transformacion.py -> solicitantes_transformed, prestamos_transformed
         |
         v
  (5) validacion.py     -> auditoria final (rompe build si falla)
```

Vamos etapa por etapa.

---
## 5. Etapa 1 — Ingesta (`scripts/ingesta.py`)

**Que hace.** Lee el CSV crudo y lo separa en las dos tablas raw (`solicitantes_raw` y `prestamos_raw`), respetando la FK.

**Idea clave.** No transforma nada. La ingesta es "espejo del CSV" — si algo viene sucio del origen, entra sucio. La unica preocupacion aqui es **fidelidad** y **idempotencia** (cada corrida vacia y vuelve a cargar, sin duplicar).

**Pasos:**
1. Leer `data/loan_data.csv` con pandas.
2. Vaciar las dos tablas raw (`TRUNCATE ... RESTART IDENTITY CASCADE`) en orden hijo->padre por la FK.
3. Insertar las 8 columnas de solicitante en `solicitantes_raw`. Postgres genera el `id` automaticamente (SERIAL).
4. Recuperar esos `id` en orden y usarlos como `solicitante_id` en las filas de `prestamos_raw`.
5. Insertar las 6 columnas de prestamo + la FK en `prestamos_raw`.

Abajo simulamos los pasos 1, 3 y 5 sobre el CSV (sin Postgres), solo para ver la separacion.

In [ ]:
COLS_SOLICITANTE = [
    "person_age", "person_gender", "person_education", "person_income",
    "person_emp_exp", "person_home_ownership", "cb_person_cred_hist_length",
    "credit_score",
]
COLS_PRESTAMO = [
    "loan_amnt", "loan_intent", "loan_int_rate", "loan_percent_income",
    "previous_loan_defaults_on_file", "loan_status",
]

solicitantes = df[COLS_SOLICITANTE].head(5).copy()
prestamos = df[COLS_PRESTAMO].head(5).copy()
prestamos.insert(0, "solicitante_id", range(1, 6))  # como lo asignaria Postgres

print("=== solicitantes_raw (primeras 5) ===")
display(solicitantes)
print("=== prestamos_raw (primeras 5) ===")
display(prestamos)

---
## 6. Etapa 2 — QualityCheck (`scripts/qualitycheck.py`)

**Que hace.** Mide la calidad de los datos *recien ingestados* (en estado raw) y emite un score 0-100 con cuatro dimensiones. **Es informativo**: nunca rompe el build (exit 0 siempre).

**Por que no rompe.** El CSV crudo se *espera* que tenga problemas (es la materia prima). El gate de calidad real esta al final (`validacion.py`). QualityCheck es **observabilidad**: si la fuente se degrada entre corridas, el score baja y nos avisa.

**Las 4 dimensiones que mide (y sus pesos):**

| Dimension | Peso | Como se detecta |
|---|---|---|
| Nulos / faltantes | 0.30 | `df.isnull().any().any()` |
| Duplicados | 0.20 | `df.duplicated().any()` |
| Outliers | 0.20 | Regla IQR (Q3 + 1.5·IQR o < Q1 − 1.5·IQR) en columnas numericas |
| Inconsistencias | 0.30 | Negativos donde no corresponde + categoricas mal normalizadas (mayusculas, espacios) |

**Formula del score:**

$$
\text{score} = (1 - \sum_i w_i \cdot \mathbb{1}[\text{dimension}_i \text{ falla}]) \times 100
$$

Es decir: parto de 100 y descuento el peso de cada dimension que falla.

**Umbrales de alerta:**
- `score >= 70` -> **OK**
- `50 <= score < 70` -> **WARNING**
- `score < 50` -> **CRITICAL**

**Resultado tipico de nuestra corrida:**
- `solicitantes_raw`: 80/100 (OK) — solo outliers.
- `prestamos_raw`: 60/100 (WARNING) — duplicados + outliers.

Eso es exactamente lo esperado: en raw hay outliers, los vamos a domar en la siguiente etapa.

In [ ]:
# Simulacion del score de calidad sobre el CSV completo
def tiene_outliers_iqr(d):
    for c in d.select_dtypes(include="number").columns:
        q1, q3 = d[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        if ((d[c] < q1 - 1.5*iqr) | (d[c] > q3 + 1.5*iqr)).any():
            return True
    return False

pesos = {"nulos": 0.30, "duplicados": 0.20, "outliers": 0.20, "inconsistencias": 0.30}

def score(d, excl_neg=()):
    checks = {
        "nulos": d.isnull().values.any(),
        "duplicados": d.duplicated().any(),
        "outliers": tiene_outliers_iqr(d),
        "inconsistencias": any(
            (d[c] < 0).any()
            for c in d.select_dtypes(include="number").columns
            if c not in excl_neg
        ),
    }
    penalty = sum(pesos[k] for k, v in checks.items() if v)
    return round((1 - penalty) * 100, 2), checks

for nombre, cols, excl in [
    ("solicitantes_raw", COLS_SOLICITANTE, ()),
    ("prestamos_raw", COLS_PRESTAMO, ("loan_status",)),
]:
    s, c = score(df[cols], excl)
    print(f"{nombre:20s}  score={s:5.2f}/100  flags={c}")

---
## 7. Etapa 3 — Limpieza (`scripts/limpieza.py`)

**Que hace.** Toma los datos raw y los deja listos para usar, aplicando reglas duras del cap. 9 del diseño tecnico. Es la etapa con mas pasos y la que define la calidad final.

**El orden de los 7 pasos NO se puede cambiar.** Cada paso asume que el anterior ya corrio:

### 7.1. Paso 1 — Extraer datos (JOIN raw)

Carga `solicitantes_raw + prestamos_raw` con un `JOIN` por la FK, ordenado por `id`. Trabajamos sobre una tabla plana para los pasos siguientes; al final volvemos a separar en las dos entidades clean.

### 7.2. Paso 2 — Remover duplicados

`df.drop_duplicates()`. Es la **unica operacion que reduce filas** en todo el pipeline. Si una fila aparece dos veces identica, vale una sola. En nuestro dataset removio 0 (el CSV no tiene duplicados exactos).

### 7.3. Paso 3 — Imputar nulos

Para no perder filas por celdas vacias aisladas:
- **Numericas:** se rellena con la **mediana** de la columna (robusta a outliers, mejor que la media).
- **Categoricas:** se rellena con la **moda** (valor mas frecuente).

En nuestro CSV: 0 nulos. Pero el paso existe por robustez (si la fuente cambia, no nos rompemos).

### 7.4. Paso 4 — Reglas de rango (clip individual)

Recorta cada variable a su rango legal:

```python
person_age          -> clip(18, 100)
person_income       -> clip(lower=0)
credit_score        -> clip(300, 850)
loan_int_rate       -> clip(5, 30)
loan_percent_income -> clip(0, 1)
loan_amnt           -> los <= 0 se reemplazan por la mediana del resto
```

**Por que `clip` y no eliminar.** `clip` preserva la fila pero ajusta el valor al limite mas cercano. Si tengo una persona con edad 150, no la borro: la dejo en 100 (la cota superior). Asi no perdemos datos por errores puntuales.

**Importante:** este paso **fija `person_age` para siempre**. Los pasos siguientes dependen de ese valor.

### 7.5. Paso 5 — Reglas cruzadas

Las invariantes que dependen de `person_age` (ya fijado en el paso anterior):

```python
person_emp_exp              -> clip(0, person_age - 18)
cb_person_cred_hist_length  -> clip(0, person_age)
```

**Por que despues del paso 4.** Si limito `person_emp_exp` antes de fijar `person_age`, puedo dejar inconsistencias (alguien con edad 200 y experiencia 50, donde luego edad pasa a 100 y experiencia sigue siendo 50 — sigue siendo coherente, pero si el orden se invierte tipico bug).

### 7.6. Paso 6 — Dominios categoricos

Para cada categorica nominal, si aparece un valor fuera del conjunto cerrado se reemplaza por la **moda** (el valor valido mas frecuente). Esto cubre tipos como `Femenino` en vez de `female`, o un `loan_intent` invalido. En nuestro dataset todo el CSV ya viene con dominios limpios — el paso esta de guardia.

Tambien se valida que `loan_status ∈ {0, 1}`; cualquier otro valor se mapea a 0.

### 7.7. Paso 7 — Winsorizacion al 5% (al final)

**Winsorizar** significa recortar los valores extremos a un percentil dado. En nuestro caso (limite 5%):
- Todo valor por debajo del percentil 5 se reemplaza por el valor del percentil 5.
- Todo valor por encima del percentil 95 se reemplaza por el valor del percentil 95.

**A que columnas se aplica:** solo a las que **no tienen rango duro** definido en el cap. 9.

| Columna | ¿Se winsoriza? | Por que |
|---|---|---|
| `person_income` | Si | No tiene tope duro definido; los ingresos altisimos distorsionan. |
| `loan_amnt` | Si | Lo mismo; montos extremos podrian sesgar. |
| `person_age` | **No** | Ya esta acotado a [18, 100] (rango duro). |
| `credit_score` | **No** | Ya esta acotado a [300, 850]. |
| `loan_int_rate` | **No** | Ya esta acotado a [5, 30]. |
| `loan_percent_income` | **No** | Ya esta acotado a [0, 1]. |

**Por que al final.** Si winsorizara `person_age` antes del paso 5, podria reducir la edad maxima a (por ejemplo) 70, pero ya habia gente con `person_emp_exp = 60` consistente con el 80 original — y se rompe la regla cruzada. Para evitarlo: rangos duros y reglas cruzadas primero, winsorizacion al final solo sobre columnas "libres".

In [ ]:
# Demo de las 4 reglas mas visibles sobre una muestra adulterada
demo = df.head(5).copy()
demo.loc[0, "person_age"] = 150         # fuera de rango
demo.loc[1, "credit_score"] = 1200      # fuera de rango
demo.loc[2, "loan_int_rate"] = 45       # fuera de rango
demo.loc[3, "person_emp_exp"] = 80      # > age-18 si age=22
print("ANTES:")
display(demo[["person_age", "credit_score", "loan_int_rate", "person_emp_exp"]])

# Aplicamos en orden: rangos, despues reglas cruzadas
demo["person_age"] = demo["person_age"].clip(18, 100)
demo["credit_score"] = demo["credit_score"].clip(300, 850)
demo["loan_int_rate"] = demo["loan_int_rate"].clip(5, 30)
demo["person_emp_exp"] = demo["person_emp_exp"].clip(lower=0, upper=demo["person_age"] - 18)

print("DESPUES:")
display(demo[["person_age", "credit_score", "loan_int_rate", "person_emp_exp"]])

In [ ]:
# Efecto de la winsorizacion al 5% sobre person_income
income = df["person_income"].copy()
lo, hi = income.quantile([0.05, 0.95])
income_w = income.clip(lo, hi)

pd.DataFrame({
    "antes": income.describe(),
    "despues_winsor5": income_w.describe(),
}).round(2)

---
## 8. Etapa 4 — Transformacion (`scripts/transformacion.py`)

**Que hace.** Toma los datos clean y genera 3 features nuevas (variables derivadas) que enriquecen la informacion del prestamo y que el EDA mostro que son las que mas correlacionan con el target. Las guarda en `prestamos_transformed`.

**Por que aqui y no en limpieza.** Limpieza arregla; transformacion enriquece. Son responsabilidades distintas.

**Las 3 features son deterministicas** — su valor depende solo de la fila, no de la distribucion. Eso significa que **no introducen data leakage** (no "miran" otras filas del set de entrenamiento). Las features que dependerian de la distribucion (escalado, encoding one-hot) se delegan al pipeline de sklearn en la fase de modelado, donde se ajustan solo con datos de entrenamiento.

### 8.1. Feature 1 — `rate_x_pct_income`

**Formula:**
$$
\text{rate\_x\_pct\_income} = \text{loan\_int\_rate} \times \text{loan\_percent\_income}
$$

**Que significa.** Es la **interaccion** entre dos cosas malas:
- una tasa alta (caro de pagar),
- y un prestamo que pesa mucho sobre el ingreso (poco margen para pagarlo).

Cuando las dos suben juntas, el riesgo se multiplica. Un prestamo con 30% de tasa **y** que vale 50% de tu ingreso anual no es "un poco peor" que uno con 15%/25%: es mucho peor. Multiplicando capturamos esa amplificacion.

**Correlacion con `loan_status`:** |corr| ~ 0.46. Es la segunda mas fuerte.

### 8.2. Feature 2 — `loan_burden`

**Formula:**
$$
\text{loan\_burden} = \frac{\text{loan\_amnt} \times (1 + \text{loan\_int\_rate}/100)}{\text{person\_income}}
$$

**Que significa.** El **costo total** del prestamo (capital + intereses simples) como fraccion del ingreso anual.

Si `loan_burden = 0.3`, significa que tendria que destinar 30% de un año entero de ingresos para pagar este prestamo. Es una medida directa de cuanto pesa la deuda sobre la economia del solicitante.

**Correlacion con `loan_status`:** |corr| ~ 0.40.

*Detalle de implementacion:* dividimos por `person_income.clip(lower=1)` para evitar division por cero. Si el ingreso fuera 0, esa fila ya habria sido marcada por las reglas de limpieza, pero protegemos por las dudas.

### 8.3. Feature 3 — `has_prev_defaults`

**Formula:**
$$
\text{has\_prev\_defaults} = \begin{cases} 1 & \text{si } \text{previous\_loan\_defaults\_on\_file} = \text{Yes} \\ 0 & \text{si no} \end{cases}
$$

**Que significa.** Solo la codificacion binaria de `previous_loan_defaults_on_file` (Yes/No → 1/0). No hay magia: pasar texto a numero para que el modelo lo pueda usar directo.

**Por que es feature derivada y no se hace en limpieza:** porque mantenemos en clean el dato tal como vino (`previous_loan_defaults_on_file = "Yes"/"No"`); el encoding es una decision de modelado.

**Correlacion con `loan_status`:** |corr| ~ 0.54. **Es la mas fuerte de todo el dataset.** Tiene sentido: alguien que ya hizo default antes, lo va a hacer de nuevo con mucha mayor probabilidad.

In [ ]:
# Calculamos las 3 features sobre el CSV y verificamos correlaciones con el target
feat = df.copy()
feat["rate_x_pct_income"] = (feat["loan_int_rate"] * feat["loan_percent_income"]).round(4)
feat["loan_burden"] = (
    (feat["loan_amnt"] * (1 + feat["loan_int_rate"] / 100))
    / feat["person_income"].clip(lower=1)
).round(4)
feat["has_prev_defaults"] = (feat["previous_loan_defaults_on_file"] == "Yes").astype(int)

feat[["rate_x_pct_income", "loan_burden", "has_prev_defaults", "loan_status"]].head()

In [ ]:
# Correlaciones con loan_status: 
# las 3 features derivadas vs las originales con peor desempeño
cols = [
    "rate_x_pct_income", "loan_burden", "has_prev_defaults",
    "loan_percent_income", "loan_int_rate",
    "credit_score", "person_age",   # las que NO sirven como features
]
correl = feat[cols + ["loan_status"]].corr(numeric_only=True)["loan_status"].drop("loan_status")
correl.abs().sort_values(ascending=False).round(3).to_frame("|corr| con loan_status")

**Lectura del cuadro de arriba (importante para la defensa):**

- Las 3 features derivadas (`has_prev_defaults`, `rate_x_pct_income`, `loan_burden`) tienen correlacion > 0.40 con el target. Son fuertes.
- Las dos originales mas predictivas (`loan_percent_income`, `loan_int_rate`) tambien sirven, pero las features derivadas las amplifican.
- `credit_score` y `person_age` aparecen con correlacion **practicamente cero** (~0.02 y ~0.008). Por eso **no se construye ninguna feature derivada a nivel solicitante** (ej: `fico_band`, `age_group`) — el EDA no las justifica.

**Esto es lo central de la decision de diseño:** las features se eligen por evidencia EDA, no por intuicion. Si el dato dice que el score crediticio no predice, no se gasta esfuerzo en derivar bandas FICO.

---
## 9. Etapa 5 — Validacion (`scripts/validacion.py`)

**Que hace.** Auditoria final sobre las 4 tablas (clean × 2 + transformed × 2). **Es el contrato de calidad de salida** — si una sola regla dura falla, el script hace `sys.exit(1)` y el CI se marca en rojo.

**Diferencia con qualitycheck:**

| | `qualitycheck.py` | `validacion.py` |
|---|---|---|
| Cuando | Despues de ingesta | Al final |
| Sobre que | Tablas `_raw` | Tablas `_clean` y `_transformed` |
| Tipo | KPI / monitoreo (suave) | Gate (duro) |
| Exit code | 0 siempre | 0 si pasa todo, 1 si falla algo |

**Que valida exactamente:**

Para **solicitantes** (clean y transformed):
- Sin nulos.
- `person_age` ∈ [18, 100], `credit_score` ∈ [300, 850].
- `person_income >= 0`.
- Reglas cruzadas: `0 <= person_emp_exp <= person_age - 18`, `0 <= cb_person_cred_hist_length <= person_age`.
- Categoricas dentro de su dominio (genero, educacion, vivienda).

Para **prestamos** (clean y transformed):
- Sin nulos.
- `loan_int_rate` ∈ [5, 30], `loan_percent_income` ∈ [0, 1], `loan_amnt > 0`.
- Categoricas dentro de su dominio (`loan_intent`, `previous_loan_defaults_on_file`).
- `loan_status ∈ {0, 1}`.

Para **prestamos_transformed**, ademas:
- `rate_x_pct_income >= 0`, `loan_burden >= 0`, `has_prev_defaults ∈ {0, 1}`.

**Detalle de acoplamiento:** `validacion.py` importa `CAT_DOMAINS` de `limpieza.py`. Esto es intencional (no quiero duplicar las constantes en dos lados), pero implica que validacion no puede correr en un contenedor sin `limpieza.py` importable.

Al final imprime el conteo: `XX defaults / YY pagados`. En nuestra corrida: 10.000 defaults / 35.000 pagados (~22% / ~78%).

---
## 10. Mapa mental: que se pasa entre etapas

| | Filas | Columnas en `solicitantes_*` | Columnas en `prestamos_*` |
|---|---|---|---|
| Despues de ingesta | 45.000 | 8 columnas originales | 6 columnas + FK |
| Despues de limpieza | 45.000* | 8 columnas (mismas, depuradas) | 6 columnas + FK (mismas, depuradas) |
| Despues de transformacion | 45.000 | 8 columnas (sin cambios) | 6 + FK + **3 features derivadas** |

*45.000 sigue siendo 45.000 porque en nuestro CSV no hubo duplicados que remover. La limpieza no elimina filas por reglas — solo recorta valores (clip, winsor) o imputa.

**Las features derivadas se agregan SOLO a `prestamos_transformed`** porque las 3 dependen de variables del prestamo (tasa, monto, % ingreso, defaults previos). `solicitantes_transformed` queda igual a `solicitantes_clean` — su existencia es para mantener el contrato del schema (raw/clean/transformed para cada entidad) aunque el EDA no haya justificado agregar columnas nuevas a la entidad solicitante.

---
## 11. Preguntas frecuentes (utiles para la defensa oral)

**1. ¿Por que separar el CSV en dos tablas si es 1-a-1?**

Porque el modelo conceptual tiene dos entidades distintas (Solicitante y Prestamo) y, en el mundo real, una persona puede pedir varios prestamos. Diseñar 1-a-N desde el inicio es buena practica, lo pide la rubrica EP2 (cap. 8), y permite escalar a multiples prestamos por solicitante sin migrar el schema.

**2. ¿Por que `clip` y no eliminar filas con valores fuera de rango?**

Para preservar volumen de datos. Si tengo 45.000 filas y elimino las 200 con anomalias, pierdo informacion legitima (las otras 13 columnas de esa fila estan bien). `clip` mantiene la fila ajustando solo el valor problematico al limite del rango legal.

**3. ¿Por que la winsorizacion va al final y solo sobre 2 columnas?**

Solo aplica donde no hay rango duro definido (`person_income`, `loan_amnt`). Si winsorizara `person_age` antes de las reglas cruzadas, podria romper la consistencia `person_emp_exp <= person_age - 18`. Ademas, las 4 columnas con rango duro ya tienen su contrato cumplido — winsorizar arriba seria destruir informacion legitima.

**4. ¿Por que solo 3 features derivadas?**

Porque el EDA mostro que `credit_score` (|corr| 0.008) y `person_age` (|corr| 0.02) NO correlacionan con `loan_status` en este dataset. Las features anteriores `fico_band` y `age_group` se eliminaron por falta de evidencia. Solo se conservaron las features cuyo respaldo cuantitativo (matriz de correlacion en `notebooks/features.ipynb`) supera |corr| 0.40.

**5. ¿Por que `qualitycheck.py` no rompe el build si encuentra problemas?**

Porque opera sobre las tablas `_raw` y los datos crudos *se espera* que tengan problemas. Su rol es **observabilidad** (alertar si la fuente se degrada entre corridas), no gate. El gate verdadero esta en `validacion.py`, que corre al final sobre las tablas `clean` y `transformed`.

**6. ¿Que es "data leakage" y como se evita?**

Es contaminar el set de entrenamiento con informacion del set de prueba (por ejemplo, calcular la media para escalar usando todo el dataset). Lo evitamos al **no hacer encoding ni escalado en transformacion**: esas operaciones se delegan al pipeline de sklearn en la fase de modelado, donde se ajustan solo con datos de entrenamiento. Las 3 features que si calculamos aqui son **deterministicas** (solo dependen de la propia fila, no de la distribucion).

**7. ¿Por que el dataset esta desbalanceado y como afecta?**

22% defaults vs 78% pagados es lo esperado en credito (la mayoria de la gente paga). En la fase de modelado importa para no usar accuracy como metrica (un modelo que prediga "pagado" siempre acertaria 78%). Hay que usar **precision, recall, F1, ROC-AUC** y considerar tecnicas de balanceo (class weights, SMOTE).

**8. ¿Que es PMBOK y como aplica aqui?**

PMBOK (Project Management Body of Knowledge) es el marco de gestion de proyectos del PMI. Aplicado al EP2: definimos alcance (solo el pipeline, sin modelo), entregables (informe + presentacion + preguntas), cronograma (5 etapas en orden), riesgos (datos sucios -> qualitycheck; cambios de schema -> down -v), y un gate de calidad final (validacion).

**9. ¿Que sigue despues de esta entrega?**

Fuera de alcance del EP2 pero documentados en `notebooks/informe.md`:
- Entrenamiento del modelo (probable: Logistic Regression de baseline + Random Forest / XGBoost).
- Evaluacion con K-fold cross-validation.
- API REST para servir predicciones.
- Monitoreo en produccion (drift de datos).

---
## 12. Cierre

Si despues de leer este notebook podes responder estas 5 preguntas sin mirar, ya tenes el dominio del proyecto:

1. ¿Que es `loan_status` y que significan sus dos valores?
2. ¿Cuales son las 2 entidades del modelo y por que se separan?
3. ¿En que orden corren las 5 etapas y que hace cada una en una linea?
4. ¿Cuales son las 3 features derivadas, cual es la mas fuerte y por que?
5. ¿Por que `qualitycheck.py` no rompe el build pero `validacion.py` si?

Material complementario en el repo:
- `notebooks/eda_solicitantes.ipynb` — EDA de la entidad Solicitante.
- `notebooks/eda_prestamos.ipynb` — EDA de la entidad Prestamo.
- `notebooks/features.ipynb` — Matriz de correlacion completa y justificacion EDA de las 3 features.
- `notebooks/informe.md` — Informe tecnico EP2 (9 indicadores de la pauta).
- `CLAUDE.md` — Convenciones del repo y decisiones de diseño.